# Emotion Circuits Discovery

This notebook reproduces the methodology from "Do LLMs 'Feel'? Emotion Circuits Discovery and Control".

We'll:
1. Discover circuits responsible for emotion expression
2. Validate these circuits
3. Demonstrate control via circuit modulation

This serves as a template for discovering circuits for other behaviors like deceptive alignment.

In [ ]:
import sys
sys.path.append('..')

import torch
from src.behavior_detection import EmotionCircuitDetector
from src.utils.data_utils import create_emotion_dataset
from src.utils.visualization import plot_circuit, plot_layer_distribution

# Set random seed for reproducibility
torch.manual_seed(42)

## 1. Initialize the Detector

We'll use GPT-2 small for this demo (faster and requires less memory).
The methodology works for larger models too.

In [ ]:
detector = EmotionCircuitDetector(
    model_name="gpt2-small",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Model loaded: {detector.model_name}")
print(f"Device: {detector.device}")
print(f"Number of layers: {detector.model.cfg.n_layers}")

## 2. Prepare Emotion Dataset

We'll create prompts that express different emotions.

In [ ]:
# Load emotion dataset
emotion_data = create_emotion_dataset()

# Display examples
for emotion, examples in emotion_data.items():
    print(f"\n{emotion.upper()}:")
    for ex in examples[:2]:
        print(f"  - {ex}")

## 3. Discover Happiness Circuit

Let's discover the circuit responsible for expressing happiness.

In [ ]:
# Discover circuit for happiness
happiness_circuit = detector.discover_emotion_circuit(
    emotion_type="happiness",
    clean_prompts=emotion_data["happiness"],
    neutral_prompts=emotion_data["neutral"],
    target_tokens=["happy", "joyful", "delighted"],
    threshold=0.5,
    prune=True
)

print("\n" + happiness_circuit.summary())

## 4. Visualize the Circuit

In [ ]:
# Plot circuit components
fig = plot_circuit(
    happiness_circuit.components,
    happiness_circuit.effects,
    title="Happiness Circuit Components",
    top_k=20
)
fig.show()

# Analyze circuit composition
stats = detector.analyze_circuit_components(happiness_circuit)
print(f"\nCircuit Statistics:")
print(f"  Total components: {stats['total_components']}")
print(f"  Layers involved: {stats['layers']}")
print(f"  Attention heads: {stats['num_attention_heads']}")
print(f"  MLP components: {stats['num_mlp']}")

## 5. Validate the Circuit

Test on held-out examples to measure faithfulness.

In [ ]:
# Create test set
test_happy = [
    "What an amazing achievement! I'm so",
    "I couldn't be happier about this! I feel",
    "This is wonderful news! I'm very"
]

test_neutral = [
    "The data shows that results are",
    "According to the schedule, we will",
    "The report indicates that numbers are"
]

# Validate
validation_results = detector.validate_emotion_circuit(
    happiness_circuit,
    test_happy,
    test_neutral,
    target_tokens=["happy", "joyful"]
)

print("\nValidation Results:")
for metric, value in validation_results.items():
    print(f"  {metric}: {value:.3f}")

## 6. Circuit Modulation: Control Emotion Expression

By amplifying or dampening the circuit, we can control how strongly the model expresses happiness.

In [ ]:
test_prompt = ["I just heard the news about the project. I feel"]

print("Testing emotion modulation:\n")

# Baseline (no modulation)
print("BASELINE (intensity=1.0):")
baseline = detector.modulate_emotion(happiness_circuit, test_prompt, intensity=1.0)
print(baseline[0])

# Amplified happiness
print("\nAMPLIFIED (intensity=2.0):")
amplified = detector.modulate_emotion(happiness_circuit, test_prompt, intensity=2.0)
print(amplified[0])

# Dampened happiness
print("\nDAMPENED (intensity=0.5):")
dampened = detector.modulate_emotion(happiness_circuit, test_prompt, intensity=0.5)
print(dampened[0])

# Removed happiness
print("\nREMOVED (intensity=0.0):")
removed = detector.modulate_emotion(happiness_circuit, test_prompt, intensity=0.0)
print(removed[0])

## 7. Discover Circuits for Multiple Emotions

Let's discover circuits for different emotions and compare them.

In [ ]:
# Discover sadness circuit
sadness_circuit = detector.discover_emotion_circuit(
    emotion_type="sadness",
    clean_prompts=emotion_data["sadness"],
    neutral_prompts=emotion_data["neutral"],
    target_tokens=["sad", "unhappy", "disappointed"],
    threshold=0.5,
    prune=True
)

# Discover anger circuit
anger_circuit = detector.discover_emotion_circuit(
    emotion_type="anger",
    clean_prompts=emotion_data["anger"],
    neutral_prompts=emotion_data["neutral"],
    target_tokens=["angry", "furious", "outraged"],
    threshold=0.5,
    prune=True
)

print("\nDiscovered all emotion circuits!")

## 8. Compare Emotion Circuits

In [ ]:
# Compare circuits
comparison = detector.compare_emotion_circuits({
    "happiness": happiness_circuit,
    "sadness": sadness_circuit,
    "anger": anger_circuit
})

print(f"Number of circuits: {comparison['num_circuits']}")
print(f"Shared components: {comparison['num_shared']}")
print(f"Circuit sizes: {comparison['circuit_sizes']}")

# Visualize layer distribution
circuits_dict = {
    "happiness": happiness_circuit.components,
    "sadness": sadness_circuit.components,
    "anger": anger_circuit.components
}

fig = plot_layer_distribution(
    circuits_dict,
    model_num_layers=detector.model.cfg.n_layers,
    title="Emotion Circuits Layer Distribution"
)
fig.show()

## Key Takeaways

1. **Circuit Discovery Works**: We successfully identified circuits for emotions
2. **Circuits are Sparse**: Only a subset of model components are crucial
3. **Control is Possible**: Modulating circuits changes behavior predictably
4. **Emotions Share Components**: Some components are shared across emotions

## Next Steps

This methodology can be adapted for:
- Deceptive alignment detection (see notebook 02)
- Power-seeking behavior (see notebook 03)
- Any other behavioral pattern you want to understand or control